# 🎓 Particle-Particle interaction


In this tutorial, we show an example of the 'particle-particle-interaction' functionality in Parcels. Note that this functionality is still fairly rudimentary for now. 

```{note}
Importantly, as particle-interaction is a many-body problem it scales as $N^2$ and can thus become very slow for large ParticleSets. There is currently **no advanced optimisation** (such as using techniques from [Smoothed Particle Hydrodynamics](https://en.wikipedia.org/wiki/Smoothed-particle_hydrodynamics)) built in to Parcels.
```

## Pulling particles

Below is an example of what can be done with particle-particle interaction. We create a square grid of $N\times N$ particles, which are all subject to Brownian Motion (via the built-in {py:func}`parcels.kernels.DiffusionUniformKh` Kernel). Furthermore, two of the particles also 'attract' other particles that are within the interaction distance: these attracted particles move with a constant velocity to the attracting particles.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from matplotlib.animation import FuncAnimation

import parcels
from parcels._datasets.structured.generated import simple_UV_dataset

# for interactive display of animations
plt.rcParams["animation.html"] = "jshtml"

In [ ]:
def Pull(particles, fieldset):
    """Kernel that "pulls" all neighbour particles
    toward the attracting particle with a constant velocity
    """
    interaction_distance = 0.5
    velocity = -0.04  # predefined attracting velocity

    # Boolean mask and coordinates of attractors
    attractor_mask = particles.attractor.astype(bool)
    x_a = particles.x[attractor_mask]
    y_a = particles.y[attractor_mask]

    # Pairwise differences and distances (n_attractors × n_particles)
    dx = particles.x - x_a[:, None]
    dy = particles.y - y_a[:, None]
    distances = np.sqrt(dx**2 + dy**2)

    # Mask dx, dy by interaction range
    within = distances < interaction_distance
    dx = np.where(within, dx, 0.0)
    dy = np.where(within, dy, 0.0)

    with np.errstate(divide="ignore", invalid="ignore"):
        inv_dist = np.where(distances > 0, 1.0 / distances, 0.0)
        dx_norm = dx * inv_dist
        dy_norm = dy * inv_dist

    particles.dx += np.sum(dx_norm, axis=0) * velocity * particles.dt
    particles.dy += np.sum(dy_norm, axis=0) * velocity * particles.dt

In [ ]:
def DiffusionFieldSet():
    """Define a fieldset with only diffusion"""
    ds = simple_UV_dataset(n_times=1, nz=1, ny=1, nx=1, mesh="flat")
    fieldset = parcels.FieldSet.from_sgrid_conventions(ds, mesh="flat")
    fieldset.add_constant_field("Kh_zonal", 0.0005)
    fieldset.add_constant_field("Kh_meridional", 0.0005)
    return fieldset

In [ ]:
npart = 11
X, Y = np.meshgrid(np.linspace(-1, 1, npart), np.linspace(-1, 1, npart))

# Create custom particle class with extra variable that indicates
# whether the interaction kernel should be executed on this particle.
InteractingParticle = parcels.Particle.add_variable(
    parcels.Variable("attractor", dtype=np.bool_),
)

attractor = [
    True if i in [int(npart * npart / 2 - 3), int(npart * npart / 2 + 3)] else False
    for i in range(npart * npart)
]

pset = parcels.ParticleSet(
    fieldset=DiffusionFieldSet(),
    pclass=InteractingParticle,
    x=X,
    y=Y,
    attractor=attractor,
)

output_file = parcels.ParticleFile(
    path="InteractingParticles.parquet",
    outputdt=np.timedelta64(1, "s"),
)

kernels = [
    parcels.kernels.DiffusionUniformKh,
    Pull,  # Add the Pull kernel defined above
]

pset.execute(
    kernels=kernels,
    runtime=np.timedelta64(60, "s"),
    dt=np.timedelta64(1, "s"),
    output_file=output_file,
    verbose_progress=False,
)

In [ ]:
df = parcels.read_particlefile("InteractingParticles.parquet")
df_attr = df.filter(pl.col("attractor") == 1)
df_other = df.filter(pl.col("attractor") == 0)

timerange = df["t"].unique()

fig = plt.figure(figsize=(4, 4), constrained_layout=True)
ax = fig.add_subplot()

ax.set_ylabel("Meridional distance [m]")
ax.set_xlabel("Zonal distance [m]")
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)

particles = df_other.filter(pl.col("t") == pl.lit(timerange[0]))
scatter = ax.scatter(particles["x"], particles["y"], c="b", s=5, zorder=1)
particles_attr = df_attr.filter(pl.col("t") == pl.lit(timerange[0]))
scatter_attr = ax.scatter(
    particles_attr["x"], particles_attr["y"], c="r", s=40, zorder=2
)
circs = []
for x_a, y_a in zip(particles_attr["x"], particles_attr["y"], strict=True):
    circs.append(
        ax.add_patch(
            plt.Circle(
                (x_a, y_a), 0.25, facecolor="None", edgecolor="r", linestyle="--"
            )
        )
    )

title = ax.set_title(
    f"Particles at t = {timerange[0].total_seconds()}s\n(Red particles are attractors)"
)


def animate(i):
    particles = df_other.filter(pl.col("t") == pl.lit(timerange[i]))
    scatter.set_offsets(np.c_[particles["x"], particles["y"]])
    particles_attr = df_attr.filter(pl.col("t") == pl.lit(timerange[i]))
    scatter_attr.set_offsets(np.c_[particles_attr["x"], particles_attr["y"]])
    for c, x_a, y_a in zip(
        circs, particles_attr["x"], particles_attr["y"], strict=True
    ):
        c.center = (x_a, y_a)
    title.set_text(
        f"Particles at t = {timerange[i].total_seconds()}s\n(Red particles are attractors)"
    )


# Create animation
anim = FuncAnimation(fig, animate, frames=len(timerange), interval=100)
plt.close(fig)
anim

## Merging particles

Another type of interaction is the merging of particles. The merging Kernel also comes with limitations (only mutual-nearest particles can be accurately merged), so this is really just a prototype. Nevertheless, the example below shows the possibilities that merging of particles can provide for more complex simulations.

In [ ]:
def Merge(particles, fieldset):
    """Kernel that "merges" two particles that are within interaction_distance range
    Particles must have a 'mass' Variable, and during merging the mass of the smallest
    particle is transferred to the largest particle. The smallest particle is then deleted.
    """
    interaction_distance = 0.05

    N = len(particles.x)

    # calculate pairwise distances (n_particles × n_particles)
    dx = particles.x[None, :] - particles.x[:, None]
    dy = particles.y[None, :] - particles.y[:, None]
    distances = np.sqrt(dx**2 + dy**2)

    # mask distances by interaction range
    within = (distances > 0) & (distances < interaction_distance)
    dist_masked = np.where(within, distances, np.inf)

    # nearest neighbour for each particle (index) and corresponding distance
    nn = np.argmin(dist_masked, axis=1)
    nn_dist = dist_masked[np.arange(N), nn]
    has_nn = np.isfinite(nn_dist)

    # mutual nearest: nn[nn[i]] == i
    i_idx = np.arange(N)
    mutual = has_nn & (nn[nn] == i_idx)

    # define pairs i and their nearest neighbour j
    pair_mask = mutual & (i_idx < nn)
    pair_i = i_idx[pair_mask]
    pair_j = nn[pair_mask]

    if pair_i.size == 0:
        return

    # compute which of the two (i, j) has the largest mass
    mass_i = particles.mass[pair_i]
    mass_j = particles.mass[pair_j]
    larger_idx = np.where(mass_j > mass_i, pair_j, pair_i)
    smaller_idx = np.where(mass_j > mass_i, pair_i, pair_j)

    # transfer mass from smaller to larger and mark smaller for deletion
    particles.mass[larger_idx] += particles.mass[smaller_idx]
    particles.state[smaller_idx] = parcels.StatusCode.Delete

In [ ]:
npart = 800

# Create custom Particle class with extra variable mass
MergeParticle = parcels.Particle.add_variable(
    parcels.Variable("mass", dtype=np.float32)
)

pset = parcels.ParticleSet(
    fieldset=DiffusionFieldSet(),
    pclass=MergeParticle,
    x=np.random.uniform(-1, 1, size=npart),
    y=np.random.uniform(-1, 1, size=npart),
    mass=np.random.uniform(0.5, 1.5, size=npart),
)

output_file = parcels.ParticleFile(
    path="MergingParticles.parquet",
    outputdt=np.timedelta64(1, "s"),
)

kernels = [
    parcels.kernels.DiffusionUniformKh,
    Merge,  # Add the Merge kernel defined above
]

pset.execute(
    kernels=kernels,
    runtime=np.timedelta64(60, "s"),
    dt=np.timedelta64(1, "s"),
    output_file=output_file,
    verbose_progress=False,
)

In [ ]:
df = parcels.read_particlefile("MergingParticles.parquet")
timerange = df["t"].unique()

fig = plt.figure(figsize=(4, 4), constrained_layout=True)
ax = fig.add_subplot()

ax.set_ylabel("Meridional distance [m]")
ax.set_xlabel("Zonal distance [m]")
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)

particles = df.filter(pl.col("t") == pl.lit(timerange[0]))
scatter = ax.scatter(
    particles["x"], particles["y"], c="b", s=particles["mass"], zorder=1
)
title = ax.set_title(f"Particles at t = {timerange[0].total_seconds()}s")


def animate(i):
    particles = df.filter(pl.col("t") == pl.lit(timerange[i]))
    scatter.set_offsets(np.c_[particles["x"], particles["y"]])
    scatter.set_sizes(particles["mass"])
    title.set_text(f"Particles at t = {timerange[i].total_seconds()}s")


anim = FuncAnimation(fig, animate, frames=len(timerange), interval=100)
plt.close(fig)
anim